<a href="https://colab.research.google.com/github/ammulu-rgb/Team5/blob/main/G_Hema_latha_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

# FACTORY PRODUCTION PLANNER (A7)
# Frontend + Backend Combined Application

%pip -q install gradio sympy numpy pandas matplotlib

import gradio as gr
import numpy as np
import sympy as sp
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Factory Production Planner is ready!")

Factory Production Planner is ready!


In [9]:

# ============================================================
# BACKEND: MATHEMATICAL ENGINE
# ============================================================

def exact_number(value):
    """Convert UI input into an exact rational number."""
    return sp.Rational(str(value))


def solve_factory(
    a11, a12, a13,
    a21, a22, a23,
    a31, a32, a33,
    h1, h2, h3
):
    # Build coefficient matrix and available-hours vector.
    # Rows = machines; columns = products.
    raw_A = [
        [a11, a12, a13],
        [a21, a22, a23],
        [a31, a32, a33]
    ]
    raw_b = [h1, h2, h3]

    try:
        A = sp.Matrix([
            [exact_number(value) for value in row]
            for row in raw_A
        ])
        b = sp.Matrix([exact_number(value) for value in raw_b])
    except Exception:
        return "## Input error\nPlease enter valid numerical values.", None

    if any(value < 0 for value in A):
        return "## Input error\nMachine-hour requirements cannot be negative.", None

    if any(value < 0 for value in b):
        return "## Input error\nAvailable machine-hours cannot be negative.", None

    products = ["Product A", "Product B", "Product C"]
    machines = ["Machine 1", "Machine 2", "Machine 3"]

    augmented = A.row_join(b)
    rank_A = A.rank()
    rank_aug = augmented.rank()

    consistent = rank_A == rank_aug

    report = [
        "# Factory Production Planner (A7)",
        "## Mathematical analysis",
        r"Model: $Ax=b$",
        "",
        f"- **Rank of A:** {rank_A}",
        f"- **Rank of augmented matrix [A | b]:** {rank_aug}",
        f"- **Consistency:** {'Consistent' if consistent else 'Inconsistent'}",
        ""
    ]

    # Display input matrix and capacity table.
    input_table = pd.DataFrame(
        [[str(v) for v in row] for row in A.tolist()],
        index=machines,
        columns=products
    )

    report.append("### Machine-hour requirements per unit")
    report.append(input_table.to_markdown())
    report.append("")
    report.append("### Reduced row-echelon form of [A | b]")
    report.append("```text\n" + str(augmented.rref()[0]) + "\n```")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # Chart 1: requirements heatmap.
    heat = np.array(A, dtype=float)
    image = axes[0].imshow(heat, cmap="Blues", aspect="auto")
    axes[0].set_xticks(range(3), products)
    axes[0].set_yticks(range(3), machines)
    axes[0].set_title("Machine Hours per Product")
    axes[0].set_xlabel("Products")
    axes[0].set_ylabel("Machines")

    for i in range(3):
        for j in range(3):
            axes[0].text(
                j, i, str(A[i, j]),
                ha="center", va="center", color="black"
            )

    fig.colorbar(image, ax=axes[0], label="Hours per unit")

    # Consistency classification and exact solution.
    if not consistent:
        report.extend([
            "## Result: No exact solution",
            "The rank theorem gives rank(A) != rank([A | b]).",
            "The machine-hour equalities cannot all be satisfied.",
            "",
            "### Conflict diagnosis"
        ])

        # Find y such that y^T A = 0 and y^T b != 0.
        certificate_found = False

        for y in A.T.nullspace():
            contradiction = (y.T * b)[0]

            if contradiction != 0:
                report.append(
                    "A mathematical inconsistency certificate exists:"
                )
                report.append(f"- Equation weights: `{list(y)}`")
                report.append(f"- Weighted left side yᵀA: `{list(y.T * A)}`")
                report.append(f"- Weighted right side yᵀb: `{contradiction}`")
                report.append(
                    "These weighted equations contradict each other. "
                    "This identifies a mathematical conflict, not automatically "
                    "the factory's physical bottleneck."
                )
                certificate_found = True
                break

        if not certificate_found:
            report.append("No inconsistency certificate was found.")

        axes[1].axis("off")
        axes[1].text(
            0.05, 0.65,
            "No exact production plan exists\nfor all supplied equalities.",
            fontsize=12,
            transform=axes[1].transAxes
        )

    else:
        solution_set = sp.linsolve((A, b))
        solution = list(next(iter(solution_set)))

        if rank_A == 3:
            report.extend([
                "## Result: Unique solution",
                "All three unknowns are determined uniquely."
            ])
        else:
            report.extend([
                "## Result: Infinitely many solutions",
                "The equations do not uniquely determine all three products.",
                "Free parameters or additional business constraints are needed."
            ])

        report.append("")
        report.append("### Production quantities")

        for name, value in zip(products, solution):
            report.append(f"- **{name}:** `{sp.simplify(value)}` units")

        free_symbols = set()
        for value in solution:
            free_symbols.update(value.free_symbols)

        # Usage chart is meaningful only for a unique numerical plan.
        if rank_A == 3 and not free_symbols:
            numeric_solution = np.array(
                [float(value) for value in solution]
            )
            numeric_used = np.array(
                [float(value) for value in A * sp.Matrix(solution)]
            )
            numeric_capacity = np.array([float(value) for value in b])

            report.append("")
            report.append("### Machine-hour verification")

            rows = []
            for i in range(3):
                rows.append({
                    "Machine": machines[i],
                    "Available hours": numeric_capacity[i],
                    "Required hours": numeric_used[i],
                    "Remaining hours": numeric_capacity[i] - numeric_used[i]
                })

            report.append(pd.DataFrame(rows).to_markdown(index=False))

            if all(value >= 0 for value in solution):
                report.append(
                    "\nAll quantities are nonnegative. The plan satisfies "
                    "the exact machine-hour equations."
                )
            else:
                report.append(
                    "\n**Warning:** at least one production quantity is negative. "
                    "The mathematical solution is not a physically feasible plan."
                )

            positions = np.arange(3)
            width = 0.35

            axes[1].bar(
                positions - width / 2,
                numeric_capacity,
                width,
                label="Available hours"
            )
            axes[1].bar(
                positions + width / 2,
                numeric_used,
                width,
                label="Required hours"
            )
            axes[1].set_xticks(positions, machines)
            axes[1].set_title("Available vs Required Hours")
            axes[1].set_ylabel("Hours")
            axes[1].legend()

        else:
            axes[1].axis("off")
            axes[1].text(
                0.05, 0.65,
                "A unique numerical production plan\ncannot be plotted.",
                fontsize=12,
                transform=axes[1].transAxes
            )

    plt.tight_layout()

    return "\n".join(report), fig

In [ ]:

# ============================================================
# FRONTEND: INTERACTIVE GRADIO APPLICATION
# ============================================================

with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown(
        """
        # 🏭 Factory Production Planner (A7)
        ### B.Tech Mathematics Mini Project

        Enter the machine-hours required per product and the
        available hours for each machine. The backend will perform
        consistency testing, rank analysis and production calculations.
        """
    )

    gr.Markdown(
        """
        **Instructions:** Each row represents a machine.
        Each column represents a product.
        The available-hours values are the right-hand side of the equations.
        """
    )

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Machine 1")
            a11 = gr.Number(value=2, label="Product A hours")
            a12 = gr.Number(value=1, label="Product B hours")
            a13 = gr.Number(value=1, label="Product C hours")
            h1 = gr.Number(value=100, label="Available hours")

        with gr.Column():
            gr.Markdown("### Machine 2")
            a21 = gr.Number(value=1, label="Product A hours")
            a22 = gr.Number(value=2, label="Product B hours")
            a23 = gr.Number(value=1, label="Product C hours")
            h2 = gr.Number(value=80, label="Available hours")

        with gr.Column():
            gr.Markdown("### Machine 3")
            a31 = gr.Number(value=1, label="Product A hours")
            a32 = gr.Number(value=1, label="Product B hours")
            a33 = gr.Number(value=2, label="Product C hours")
            h3 = gr.Number(value=90, label="Available hours")

    calculate_btn = gr.Button(
        "Calculate Production Plan",
        variant="primary"
    )

    gr.Markdown("## Results Dashboard")

    result_text = gr.Markdown()
    result_plot = gr.Plot(label="Production Analysis Charts")

    calculate_btn.click(
        fn=solve_factory,
        inputs=[
            a11, a12, a13,
            a21, a22, a23,
            a31, a32, a33,
            h1, h2, h3
        ],
        outputs=[result_text, result_plot]
    )

    gr.Markdown(
        """
        ---
        **Mathematics used:** Linear equations, augmented matrices,
        reduced row-echelon form, matrix rank and the
        Rouché–Capelli consistency theorem.

        **Model limitation:** This version solves exact equalities.
        A practical factory optimizer also needs demand, capacity
        inequalities, nonnegative quantities and potentially integer
        production constraints.
        """
    )

app.launch(share=True, debug=True)

/tmp/ipykernel_696/990557516.py:5: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://61ac2e3ae322473205.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
